In [1]:


import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import precision_score, recall_score, f1_score

year_2007 = pd.read_csv('/content/drive/MyDrive/ResearchAssistant/2007.csv')
print("Data is imported.")


def ArrDelay_Yes(row):
    if row['ArrDelay'] > 0:
        return 1  # Delayed
    else:
        return 0  # On time

year_2007['ArrDelay_Binary'] = year_2007.apply(ArrDelay_Yes, axis=1) #axis =0 is rows and axis=1 is columns

print("ArrDelay_Yes function is executed.")

#Creating function to identify Time Period

year_2007['ArrTime_Trim']= year_2007['ArrTime'].astype(str).str.strip('.0')

def categorize_time_period(hour):
  try:
    hour = int(hour)
    if 6 <= hour < 12:
      return 'Morning'
    elif 12 <= hour < 18:
      return 'Afternoon'
    elif 18 <= hour < 21:
      return 'Evening'
    else:
      return 'Night'
  except Exception as ex:
    pass

#Creating Tine Period Column
temp = []
for i in year_2007['ArrTime_Trim'].to_list():
  if len(str(i)) == 3:
    temp.append(categorize_time_period(i[:1]))
  elif len(str(i)) == 4:
    temp.append(categorize_time_period(i[:2]))

temp

temp1=pd.DataFrame(temp,columns=['DepCategory'])
year_2007['TimePeriod']=temp1
year_2007['TimePeriod'].dropna(inplace=True)

print("Year 2007 ArrCategory created.")

#Defining features

features = ['DayOfWeek', 'Month', 'Distance', 'TimePeriod', 'AirTime']
X = year_2007[features]
y = year_2007['ArrDelay_Binary']

#Handling Categorical variables
X = pd.get_dummies(X, columns=['DayOfWeek', 'Month', 'TimePeriod'])

# Handling missing values
imputer = SimpleImputer(strategy='mean')  # Replace with your preferred strategy
X_imputed = imputer.fit_transform(X)

# Splitting data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_imputed, y, test_size=0.2, random_state=42)


# Trainning the model
rf_classifier = RandomForestClassifier(n_estimators=10, random_state=42)
rf_classifier.fit(X_train, y_train)

print("Model created.")

# Making predictions
y_pred = rf_classifier.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
confusion_mat = confusion_matrix(y_test, y_pred)
classification_rep = classification_report(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("Confusion Matrix:\n", confusion_mat)
print("Classification Report:\n", classification_report)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)


Data is imported.
ArrDelay_Yes function is executed.
Year 2007 ArrCategory created.
Model created.
Accuracy: 0.5783926802057904
Confusion Matrix:
 [[506484 292069]
 [336397 355693]]
Classification Report:
 <function classification_report at 0x7a70a7125f30>
Precision: 0.5491106301388473
Recall: 0.5139403834761375
F1-score: 0.5309437161716367
